# Cell Shape Visuals — Snapshot & Collision Intensity Plots

Produces two figure types for a given set of simulations:
1. **Filled cell shape snapshots** — overlaid at regular timestep intervals, coloured by time (viridis), with Gaussian smoothing and track boundary overlay
2. **Collision intensity over time** — time-coloured line/area plot showing collision intensity for each simulation

Works with either **Track A** or **Track B** via the `TRACK` configuration parameter.

In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================

TRACK = "A"  # "A" (Track A) or "B" (Track B)

# Simulations to visualize: {model_label: simulation_id}
# Ensure these exist in your data/sample_simulations directory
SIM_IDS = {"WP": "1131", "WPI": "11"}  # Example for Track A
# SIM_IDS = {"WP": "1", "WPI": "2"}  # Example for Track B

# Plot parameters
TIMESTEP_INTERVAL = 120   # plot cell shape every N timesteps
ALPHA = 0.6               # cell shape opacity (0.0 to 1.0)
GAUSSIAN_SIGMA = 1        # smoothing for cell masks (0=none, 1-5=light to heavy)

# Exclusions
IGNORE_MODEL_TYPES = []   # e.g., ["RacProtrusion", "RacProtrusionNoise"]
IGNORE_SIM_IDS = []       # e.g., ["1205", "31"]

# ============================================================================
# PATHS (Self-contained in repository)
# ============================================================================

import os
from pathlib import Path

# Assuming we are running this inside notebooks/
REPO_ROOT = Path(os.getcwd()).parent
DATA_DIR = REPO_ROOT / "data" / "sample_simulations"
PROCESSED_DIR = REPO_ROOT / "output"
OUTPUT_DIR = REPO_ROOT / "output" / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRACK_CONFIGS = {
    "A": {
        "domain_tiff_name": "TrackA_300x450.tiff",
    },
    "B": {
        "domain_tiff_name": "TrackB_300x450.tiff",
    },
}

CFG = TRACK_CONFIGS[TRACK]

print(f"Track: {TRACK}")
print(f"Data dir: {DATA_DIR}")
print(f"Processed dir: {PROCESSED_DIR}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Simulations: {SIM_IDS}")

In [ ]:
# ============================================================================
# IMPORTS & DATA LOADING
# ============================================================================

import sys
import re
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.collections import PolyCollection, LineCollection
import cv2
from typing import List, Tuple

# Add repo root so we can import utils/
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from utils.image_processing import detect_black_pixels, analyze_png_colors
from utils.simulation_parser import find_simulation_pngs

# Font size constants
TITLE_FONTSIZE = 22
SUBTITLE_FONTSIZE = 18
AXIS_LABEL_FONTSIZE = 16
TICK_LABEL_FONTSIZE = 14

# Load processed collision data (run 01_bulk_collision_analysis.ipynb first!)
summary_path = PROCESSED_DIR / "collision_analysis_summary.pkl"
detailed_path = PROCESSED_DIR / "collision_analysis_detailed.pkl"

if not summary_path.exists() or not detailed_path.exists():
    raise FileNotFoundError(f"Processed data not found in {PROCESSED_DIR}. Please run 01_bulk_collision_analysis.ipynb first.")

summary_df = pd.read_pickle(summary_path)
detailed_df = pd.read_pickle(detailed_path)

# Ensure string IDs for consistent matching
summary_df["simulation_id"] = summary_df["simulation_id"].astype(str)
detailed_df["simulation_id"] = detailed_df["simulation_id"].astype(str)

# Apply exclusions
if IGNORE_MODEL_TYPES:
    summary_df = summary_df[~summary_df["model_type"].isin(IGNORE_MODEL_TYPES)].copy()
if IGNORE_SIM_IDS:
    summary_df = summary_df[~summary_df["simulation_id"].isin(IGNORE_SIM_IDS)].copy()
    detailed_df = detailed_df[~detailed_df["simulation_id"].isin(IGNORE_SIM_IDS)].copy()

print(f"Loaded {len(summary_df)} simulations, {len(detailed_df)} detailed frames")
print(f"Model types: {sorted(summary_df['model_type'].dropna().unique())}")

In [ ]:
# ============================================================================
# CORE UTILITY FUNCTIONS
# ============================================================================


def find_simulation_folder(simulation_id: str) -> str:
    """Find the folder for a simulation ID in the data directory."""
    for folder in DATA_DIR.iterdir():
        if folder.is_dir() and folder.name.endswith(f"_{simulation_id}"):
            return str(folder)
    raise ValueError(f"Could not find folder for simulation {simulation_id} in {DATA_DIR}")


def load_domain_image(tiff_path):
    """Load a domain TIFF and return it as an RGB numpy array."""
    if not tiff_path or not Path(tiff_path).exists():
        return None
    tiff_path = str(tiff_path)
    try:
        img = cv2.imread(tiff_path)
        if img is None:
            from PIL import Image
            with Image.open(tiff_path) as pil_img:
                arr = np.array(pil_img)
                if arr.ndim == 2:
                    img = cv2.cvtColor(arr, cv2.COLOR_GRAY2RGB)
                elif arr.shape[2] == 4:
                    img = arr[:, :, :3]
                else:
                    img = arr
                if img.dtype == np.uint16:
                    img = (img / 256).astype(np.uint8)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        return img
    except Exception as e:
        print(f"Error loading domain image {tiff_path}: {e}")
        return None


def get_domain_tiff_path(folder_path: str) -> str:
    """Find the domain TIFF file in a simulation folder or globally in DATA_DIR."""
    folder = Path(folder_path)

    # Try the known domain filename first
    known = folder / CFG["domain_tiff_name"]
    if known.exists():
        return str(known)

    # Search globally in DATA_DIR for the required TIFF
    for p in DATA_DIR.rglob(CFG["domain_tiff_name"]):
        return str(p)

    # Search for domain-related TIFF files
    for pattern in ["*domain*.tif*", "*Domain*.tif*"]:
        matches = list(folder.glob(pattern))
        if matches:
            return str(matches[0])

    # Fall back to any TIFF
    tiffs = list(folder.glob("*.tif*"))
    if tiffs:
        return str(tiffs[0])
    return None


def extract_domain_boundary(domain_image_path, domain_width, domain_height):
    """Extract a binary mask of track walls (black pixels) from the domain image."""
    if not domain_image_path or not Path(domain_image_path).exists():
        return None
    domain_img = load_domain_image(domain_image_path)
    if domain_img is None:
        return None
    if domain_img.ndim == 3:
        gray = cv2.cvtColor(domain_img, cv2.COLOR_RGB2GRAY)
    else:
        gray = domain_img
    _, boundary_mask = cv2.threshold(gray, 30, 255, cv2.THRESH_BINARY_INV)
    return boundary_mask


# --------------------------------------------------------------------------
# Cell shape extraction from PNG frames
# --------------------------------------------------------------------------

def extract_cell_shape_data(simulation_id: str,
                            domain_width: int,
                            domain_height: int,
                            time_interval: int = 1,
                            sampling_density: int = 2000) -> pd.DataFrame:
    """
    Extract cell pixel coordinates from simulation PNG frames.

    Reads each PNG, detects black (cell) pixels, samples them, and records
    coordinates scaled strictly to domain space based on image dimensions.

    Returns DataFrame with columns:
        simulation_id, model_type, timestep, frame_index, x, y, time_fraction
    """
    sim_row = summary_df[summary_df["simulation_id"] == str(simulation_id)]
    if sim_row.empty:
        raise ValueError(f"Simulation ID {simulation_id} not found in summary data")

    folder_path = find_simulation_folder(simulation_id)
    model_type = sim_row["model_type"].iloc[0]
    print(f"Extracting cell data for sim {simulation_id} ({model_type})")

    png_files = find_simulation_pngs(folder_path)
    if not png_files:
        raise ValueError(f"No PNG files found in {folder_path}")

    min_time = min(t for t, _ in png_files)
    max_time = max(t for t, _ in png_files)
    time_range = max_time - min_time if max_time > min_time else 1

    selected = png_files[::time_interval]
    print(f"Processing {len(selected)} of {len(png_files)} frames")

    cell_data = []
    for i, (timestep, filepath) in enumerate(selected):
        if i % 20 == 0:
            print(f"  Frame {i}/{len(selected)} (timestep {timestep})")
        try:
            image = cv2.imread(filepath)
            if image is None:
                continue
            black_mask = detect_black_pixels(image, tolerance=10)
            coords = np.where(black_mask == 255)
            if len(coords[0]) == 0:
                continue

            h, w = black_mask.shape
            time_fraction = (timestep - min_time) / time_range

            # Adaptive sampling
            rate = max(1, len(coords[0]) // sampling_density)
            y_indices = coords[0][::rate]
            x_indices = coords[1][::rate]

            scale_x = domain_width / w
            scale_y = domain_height / h

            for y_idx, x_idx in zip(y_indices, x_indices):
                cell_data.append({
                    "simulation_id": str(simulation_id),
                    "model_type": model_type,
                    "timestep": timestep,
                    "frame_index": i,
                    "x": float(x_idx) * scale_x,
                    "y": float(h - y_idx) * scale_y,  # flip Y: 0,0 at bottom-left
                    "time_fraction": time_fraction,
                })
        except Exception as e:
            print(f"  Error on frame {timestep}: {e}")

    if not cell_data:
        raise ValueError(f"No valid cell data from simulation {simulation_id}")

    df = pd.DataFrame(cell_data)
    print(f"Extracted {len(df)} points across {df['timestep'].nunique()} timesteps")
    return df


# --------------------------------------------------------------------------
# Persistent cache manager
# --------------------------------------------------------------------------

def get_or_process_cell_data(simulation_id, force_reprocess=False,
                             sampling_density=2000, time_interval=1):
    """
    Load cell data from the track-specific cache, or extract from PNGs if missing.

    Cache stores properly SCALED data.
    """
    cache_path = PROCESSED_DIR / "processed_cell_data_cache.pkl"

    sim_row = summary_df[summary_df["simulation_id"] == str(simulation_id)]
    if sim_row.empty:
        raise ValueError(f"Simulation {simulation_id} not found in summary_df")
    
    # Get domain dims, fallback to 300x450 if not in summary_df
    dw = 300
    dh = 450
    if "domain_width" in sim_row.columns and not pd.isna(sim_row["domain_width"].iloc[0]):
        dw = int(float(sim_row["domain_width"].iloc[0]))
    if "domain_height" in sim_row.columns and not pd.isna(sim_row["domain_height"].iloc[0]):
        dh = int(float(sim_row["domain_height"].iloc[0]))

    # Try cache
    cache = pd.DataFrame()
    if cache_path.exists() and not force_reprocess:
        try:
            cache = pd.read_pickle(cache_path)
            print(f"Cache: {cache['simulation_id'].nunique()} sims, {len(cache)} points")
        except Exception as e:
            print(f"Warning: could not load cache: {e}")
            cache = pd.DataFrame()

    # Check if sim is cached
    if not cache.empty and "simulation_id" in cache.columns and not force_reprocess:
        sim_data = cache[cache["simulation_id"] == str(simulation_id)]
        if not sim_data.empty:
            print(f"Sim {simulation_id} found in cache")
            return sim_data.copy()

    # Extract fresh
    print(f"Processing simulation {simulation_id} from PNGs...")
    cell_df = extract_cell_shape_data(
        simulation_id,
        domain_width=dw,
        domain_height=dh,
        time_interval=time_interval,
        sampling_density=sampling_density,
    )

    if cell_df is not None and not cell_df.empty:
        # Update cache with scaled data
        if cache.empty:
            cache = cell_df.copy()
        else:
            if "simulation_id" in cache.columns:
                cache = cache[cache["simulation_id"] != str(simulation_id)]
            cache = pd.concat([cache, cell_df.copy()], ignore_index=True)
        try:
            cache.to_pickle(cache_path)
            print(f"Cache updated: {cache['simulation_id'].nunique()} sims")
        except Exception as e:
            print(f"Warning: could not save cache: {e}")

        return cell_df.copy()

    return cell_df


# --------------------------------------------------------------------------
# Cell mask creation
# --------------------------------------------------------------------------

def create_cell_mask_from_points(cell_points, domain_width, domain_height,
                                 gaussian_sigma=0):
    """
    Create a filled binary mask from cell pixel coordinates.

    Uses morphological closing to fill gaps, then optional Gaussian smoothing.
    """
    mask = np.zeros((int(domain_height), int(domain_width)), dtype=np.uint8)

    if isinstance(cell_points, pd.DataFrame):
        x = cell_points["x"].values
        y = cell_points["y"].values
    else:
        x, y = cell_points[:, 0], cell_points[:, 1]

    if len(x) == 0:
        return mask

    # Convert to image coords (y=0 at top)
    px = x.astype(np.int32)
    py = (domain_height - y).astype(np.int32)

    # Filter in-bounds
    valid = (px >= 0) & (px < domain_width) & (py >= 0) & (py < domain_height)
    px, py = px[valid], py[valid]

    if len(px) == 0:
        return mask

    # Set pixels
    mask[py, px] = 255

    # Morphological closing to fill small gaps
    kernel = np.ones((3, 3), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)

    # Fill holes via contour detection
    contours, _ = cv2.findContours(mask.copy(), cv2.RETR_EXTERNAL,
                                   cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        cv2.drawContours(mask, contours, -1, 255, thickness=cv2.FILLED)

    # Optional Gaussian smoothing
    if gaussian_sigma > 0:
        mask_f = mask.astype(np.float32) / 255.0
        mask_f = cv2.GaussianBlur(mask_f, (0, 0), gaussian_sigma)
        mask = (mask_f > 0.5).astype(np.uint8) * 255

    return mask


print("Utility functions loaded.")

In [ ]:
# ============================================================================
# SNAPSHOT PLOT: Filled cell shapes coloured by time
# ============================================================================


def plot_filled_snapshots(cell_df,
                          background_image_path=None,
                          output_path=None,
                          timestep_interval=50,
                          alpha=0.5,
                          domain_width=300,
                          domain_height=450,
                          overlay_boundary=True,
                          gaussian_sigma=0):
    """
    Plot filled cell shapes at discrete timesteps, coloured by time (viridis).

    Uses alpha compositing so overlapping regions blend properly.
    The domain boundary is overlaid on top in black.
    """
    if cell_df.empty:
        raise ValueError("Empty cell data")

    fig, ax = plt.subplots(figsize=(10, 15))
    sim_id = cell_df["simulation_id"].iloc[0]
    model_type = cell_df["model_type"].iloc[0]

    # Background
    if background_image_path and Path(background_image_path).exists():
        bg = load_domain_image(background_image_path)
        if bg is not None:
            ax.imshow(bg, extent=(0, domain_width, 0, domain_height), alpha=0.3)

    # Select timesteps
    all_ts = sorted(cell_df["timestep"].unique())
    selected_ts = all_ts[::timestep_interval]

    if not selected_ts:
        print("Warning: no timesteps selected")
        return None

    print(f"Plotting {len(selected_ts)} filled snapshots for {model_type} sim {sim_id}")

    colormap = plt.cm.viridis
    dh, dw = int(domain_height), int(domain_width)
    composite = np.zeros((dh, dw, 4), dtype=np.float32)

    for i, ts in enumerate(selected_ts):
        ts_data = cell_df[cell_df["timestep"] == ts]
        if ts_data.empty:
            continue

        # Time fraction for colour
        if "time_fraction" in ts_data.columns:
            tf = ts_data["time_fraction"].iloc[0]
        else:
            tf = i / max(len(selected_ts) - 1, 1)

        color = colormap(tf)
        mask = create_cell_mask_from_points(ts_data, dw, dh, gaussian_sigma)
        mask_f = mask.astype(np.float32) / 255.0

        # Alpha compositing ("over" operator)
        src_a = mask_f * alpha
        src_rgb = np.stack([color[0] * src_a, color[1] * src_a, color[2] * src_a], axis=-1)
        dst_a = composite[:, :, 3]
        dst_rgb = composite[:, :, :3]

        out_a = src_a + dst_a * (1 - src_a)
        out_a_safe = np.where(out_a > 0, out_a, 1)
        out_rgb = (src_rgb + dst_rgb * dst_a[..., None] * (1 - src_a)[..., None]) / out_a_safe[..., None]

        composite[:, :, :3] = out_rgb
        composite[:, :, 3] = out_a

    # Display composite
    ax.imshow(composite, extent=(0, domain_width, 0, domain_height),
              origin="upper", interpolation="bilinear", zorder=5)

    # Overlay domain boundary
    if overlay_boundary and background_image_path and Path(background_image_path).exists():
        bmask = extract_domain_boundary(background_image_path, dw, dh)
        if bmask is not None:
            overlay = np.zeros((bmask.shape[0], bmask.shape[1], 4), dtype=np.uint8)
            overlay[bmask > 0] = [0, 0, 0, 255]
            ax.imshow(overlay, extent=(0, domain_width, 0, domain_height),
                      interpolation="nearest", zorder=10)

    # Colorbar
    norm = Normalize(vmin=0, vmax=1)
    sm = cm.ScalarMappable(cmap=colormap, norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, label="Time Progression")
    cbar.ax.tick_params(labelsize=TICK_LABEL_FONTSIZE)

    ax.set_title(f"Cell Snapshots \u2014 {model_type}, Sim {sim_id}\n"
                 f"(every {timestep_interval} timesteps, \u03c3={gaussian_sigma})",
                 fontsize=TITLE_FONTSIZE)
    ax.set_xlabel("X Position", fontsize=AXIS_LABEL_FONTSIZE)
    ax.set_ylabel("Y Position", fontsize=AXIS_LABEL_FONTSIZE)
    ax.tick_params(axis="both", which="major", labelsize=TICK_LABEL_FONTSIZE)
    ax.set_xlim(0, domain_width)
    ax.set_ylim(0, domain_height)

    plt.tight_layout()
    if output_path:
        plt.savefig(output_path, dpi=300, bbox_inches="tight")
        print(f"Saved: {output_path}")
        plt.close()
    else:
        plt.show()

    return fig, ax


print("Snapshot plot function loaded.")

In [ ]:
# ============================================================================
# COLLISION INTENSITY PLOT: Time-coloured line + fill
# ============================================================================


def plot_collision_intensity(sim_id,
                             output_path=None,
                             y_max_override=None):
    """
    Plot collision intensity over time for a single simulation.

    Uses viridis colouring matched to the snapshot plots:
    - Filled polygon under the curve (PolyCollection, alpha=0.4)
    - Darker viridis line on top (LineCollection)

    Args:
        sim_id: Simulation ID string
        output_path: If set, save figure to this path
        y_max_override: If set, fix the y-axis upper limit (useful for
                        cross-simulation comparisons)
    """
    sim_collision = detailed_df[detailed_df["simulation_id"] == str(sim_id)].sort_values("timestep")
    if sim_collision.empty:
        # Try numeric match
        try:
            sim_collision = detailed_df[detailed_df["simulation_id"] == int(sim_id)].sort_values("timestep")
        except (ValueError, TypeError):
            pass
    if sim_collision.empty:
        print(f"No collision data for sim {sim_id}")
        return None

    # Get model type from summary
    sim_row = summary_df[summary_df["simulation_id"] == str(sim_id)]
    model_type = sim_row["model_type"].iloc[0] if not sim_row.empty else "Unknown"

    ts = sim_collision["timestep"].values
    intensity = sim_collision["collision_intensity"].values

    ts_min, ts_max = ts.min(), ts.max()
    if ts_max > ts_min:
        time_fracs = (ts - ts_min) / (ts_max - ts_min)
    else:
        time_fracs = np.zeros_like(ts, dtype=float)

    colormap = plt.cm.viridis

    fig, ax = plt.subplots(figsize=(10, 4))

    # Filled polygon segments coloured by time
    verts, colors_list = [], []
    for i in range(len(ts) - 1):
        verts.append([(ts[i], 0), (ts[i], intensity[i]),
                      (ts[i+1], intensity[i+1]), (ts[i+1], 0)])
        mid_frac = (time_fracs[i] + time_fracs[i+1]) / 2
        colors_list.append(colormap(mid_frac))

    poly = PolyCollection(verts, facecolors=colors_list, alpha=0.4,
                          edgecolors="none", zorder=1)
    ax.add_collection(poly)

    # Darker line on top
    points = np.column_stack([ts, intensity]).reshape(-1, 1, 2)
    segments = np.concatenate([points[:-1], points[1:]], axis=1)
    seg_colors = []
    for i in range(len(time_fracs) - 1):
        mid_frac = (time_fracs[i] + time_fracs[i+1]) / 2
        c = colormap(mid_frac)
        seg_colors.append([c[0]*0.7, c[1]*0.7, c[2]*0.7, 1.0])

    lc = LineCollection(segments, colors=seg_colors, linewidths=2, zorder=5)
    ax.add_collection(lc)

    ax.set_xlim(ts_min, ts_max)
    if y_max_override is not None:
        ax.set_ylim(0, y_max_override * 1.1)
    else:
        ax.set_ylim(0, max(intensity.max() * 1.1, 1e-6))

    ax.axhline(y=0, color="black", linestyle="--", alpha=0.5, linewidth=1)
    ax.set_xlabel("Timestep", fontsize=AXIS_LABEL_FONTSIZE)
    ax.set_ylabel("Collision Intensity", fontsize=AXIS_LABEL_FONTSIZE)
    ax.set_title(f"Collision Intensity \u2014 {model_type}, Sim {sim_id}",
                 fontsize=SUBTITLE_FONTSIZE)
    ax.tick_params(axis="both", which="major", labelsize=TICK_LABEL_FONTSIZE)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    if output_path:
        plt.savefig(output_path, dpi=300, bbox_inches="tight")
        print(f"Saved: {output_path}")
        plt.close()
    else:
        plt.show()

    return fig, ax


print("Collision intensity plot function loaded.")

In [ ]:
# ============================================================================
# GENERATE FIGURES
# ============================================================================


def generate_filled_snapshot_figures(model_sim_ids,
                                    timestep_interval=100,
                                    alpha=0.6,
                                    gaussian_sigma=1):
    """
    Generate filled cell snapshot figures for each simulation.

    For each model/sim pair:
      1. Loads cell data (from cache or PNGs)
      2. Finds the domain TIFF for boundary overlay
      3. Plots filled snapshots coloured by time
      4. Saves to OUTPUT_DIR/Track{TRACK}_{model}_sim{id}_snapshots.png
    """
    for model, sim_id in model_sim_ids.items():
        print(f"\n{'='*60}")
        print(f"Generating snapshot: {model} (sim {sim_id})")
        print(f"{'='*60}")

        sim_row = summary_df[summary_df["simulation_id"] == str(sim_id)]
        if sim_row.empty:
            print(f"Warning: sim {sim_id} not found, skipping")
            continue

        folder_path = find_simulation_folder(sim_id)
        
        dw = 300
        dh = 450
        if "domain_width" in sim_row.columns and not pd.isna(sim_row["domain_width"].iloc[0]):
            dw = int(float(sim_row["domain_width"].iloc[0]))
        if "domain_height" in sim_row.columns and not pd.isna(sim_row["domain_height"].iloc[0]):
            dh = int(float(sim_row["domain_height"].iloc[0]))

        domain_tiff = get_domain_tiff_path(folder_path)
        cell_df = get_or_process_cell_data(sim_id)

        if cell_df is None or cell_df.empty:
            print(f"No data for sim {sim_id}")
            continue

        out = OUTPUT_DIR / f"Track{TRACK}_{model}_sim{sim_id}_snapshots.png"
        plot_filled_snapshots(
            cell_df,
            background_image_path=domain_tiff,
            output_path=out,
            timestep_interval=timestep_interval,
            alpha=alpha,
            domain_width=dw,
            domain_height=dh,
            gaussian_sigma=gaussian_sigma,
        )


def generate_collision_intensity_figures(model_sim_ids,
                                         y_max_override=None):
    """
    Generate collision intensity over time figures for each simulation.

    Saves to OUTPUT_DIR/Track{TRACK}_{model}_sim{id}_collision.png
    """
    for model, sim_id in model_sim_ids.items():
        print(f"\nGenerating collision plot: {model} (sim {sim_id})")
        out = OUTPUT_DIR / f"Track{TRACK}_{model}_sim{sim_id}_collision.png"
        plot_collision_intensity(sim_id, output_path=out,
                                y_max_override=y_max_override)


# ---- Run ----
generate_filled_snapshot_figures(SIM_IDS,
                                timestep_interval=TIMESTEP_INTERVAL,
                                alpha=ALPHA,
                                gaussian_sigma=GAUSSIAN_SIGMA)

generate_collision_intensity_figures(SIM_IDS)